# Cài đặt thư viện

In [ ]:
!pip install -U google-generativeai sentence-transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.13.0:
      Successfully uninstalled accelerate-1.13.0
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.5.1
    Uninstalling sentence-transformers-5.5.1:
      Successfully uninstalled sent

# Upload File

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving chunks_train.json to chunks_train.json


# Sinh dữ liệu huấn luyện

In [ ]:
import os
import json
import time
import google.generativeai as genai

API_KEYS = [
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER",
    "GEMINI_KEY_PLACEHOLDER"
]
# Số lượng chunk tối đa muốn xử lý trong lượt chạy này (mỗi chunk sinh ra 3 câu hỏi)
LIMIT = 22
# Tên file lưu trữ kết quả đầu ra
OUTPUT_PATH = 'train_embedding_data.json'

# Tự động nạp thêm key từ Colab Secrets (nếu có)
try:
    from google.colab import userdata
    env_keys = [v for k, v in os.environ.items() if 'GEMINI' in k.upper() or 'GOOGLE_API' in k.upper()]
    for k in env_keys:
        if k not in API_KEYS:
            API_KEYS.append(k)
    colab_secret = userdata.get('GEMINI_API_KEY')
    if colab_secret and colab_secret not in API_KEYS:
        API_KEYS.append(colab_secret)
except Exception:
    pass

# Nếu chưa có key nào thì yêu cầu nhập thủ công
if not API_KEYS:
    manual_key = input("Không tìm thấy API Key. Nhập Gemini API Key của bạn: ").strip()
    if manual_key:
        API_KEYS.append(manual_key)
if not API_KEYS:
    raise ValueError("Vui lòng cung cấp ít nhất 1 Gemini API Key để thực hiện sinh dữ liệu!")
print(f"Đã nạp {len(API_KEYS)} API Key cho hệ thống.")

# Lớp Client hỗ trợ tự động xoay vòng key khi gặp lỗi 429
class RobustGeminiGenerator:
    def __init__(self, api_keys, model_name='gemini-2.5-flash'):
        self.api_keys = api_keys
        self.current_key_idx = 0
        self.model_name = model_name
        self._set_model()

    def _set_model(self):
        genai.configure(api_key=self.api_keys[self.current_key_idx])
        self.model = genai.GenerativeModel(self.model_name)

    def generate(self, prompt):
        attempts = 0
        max_attempts = len(self.api_keys) * 2
        while attempts < max_attempts:
            try:
                response = self.model.generate_content(prompt)
                return response.text
            except Exception as e:
                err_str = str(e).upper()
                if "429" in err_str or "RESOURCE_EXHAUSTED" in err_str:
                    attempts += 1
                    next_idx = (self.current_key_idx + 1) % len(self.api_keys)
                    print(f"\n[CẢNH BÁO] Key số {self.current_key_idx + 1} cạn kiệt Token (Lỗi 429). Chuyển sang dùng Key số {next_idx + 1}...")
                    self.current_key_idx = next_idx
                    self._set_model()
                    time.sleep(1)
                else:
                    raise e
        raise Exception("Tất cả các API Keys trong danh sách đã bị cạn kiệt token!")

# Đọc file chunks_export.json
json_path = 'chunks_train.json'
if not os.path.exists(json_path):
    raise FileNotFoundError(f"Không tìm thấy file {json_path}")

with open(json_path, 'r', encoding='utf-8') as f:
    chunks_data = json.load(f)

existing_pairs = []
generated_chunk_ids = set()

if os.path.exists(OUTPUT_PATH):
    try:
        with open(OUTPUT_PATH, 'r', encoding='utf-8') as f:
            existing_pairs = json.load(f)
        for pair in existing_pairs:
            if 'chunk_id' in pair:
                generated_chunk_ids.add(pair['chunk_id'])
        print(f"Tìm thấy dữ liệu cũ: {len(existing_pairs)} câu hỏi (đã xử lý {len(generated_chunk_ids)} chunks).")
    except Exception as e:
        print(f"Không thể đọc file dữ liệu cũ, tiến hành ghi đè mới: {e}")

generator = RobustGeminiGenerator(API_KEYS)
new_pairs = []
processed_in_this_run = 0

print(f"\nBắt đầu sinh câu hỏi...")

for idx, item in enumerate(chunks_data):
    if processed_in_this_run >= LIMIT:
        print(f"Đã đạt giới hạn {LIMIT} chunks mới cho lượt này. Dừng tiến trình sinh dữ liệu.")
        break

    chunk_id = item.get('chunk_id')

    # Bỏ qua nếu chunk này đã được sinh câu hỏi ở đợt trước
    if chunk_id in generated_chunk_ids:
        continue

    enhanced_content = item.get('enhanced_content', '')
    doc_name = item.get('metadata', {}).get('document_name', 'Unknown')

    prompt = f"""You are an expert AI dataset generator.
Your task is to read the following text chunk from an academic paper and generate exactly 3 diverse, search-optimized search queries (questions) in English that a researcher would type to find this specific chunk.

Instructions:
1. The questions must be answered directly by the text chunk.
2. The questions should be natural, varied in phrasing, and use technical terms from the chunk.
3. Output ONLY a valid JSON list of strings, with no markdown formatting, no code blocks (like ```json), and no extra text.

Text Chunk:
{enhanced_content}

Output JSON:"""

    success = False
    attempts = 0
    while not success and attempts < 3:
        try:
            time.sleep(1.5) # Tránh bị spam limit của API
            response_text = generator.generate(prompt).strip()

            # Làm sạch phản hồi JSON
            if response_text.startswith("```json"):
                response_text = response_text[7:]
            if response_text.endswith("```"):
                response_text = response_text[:-3]
            response_text = response_text.strip()

            queries = json.loads(response_text)
            if isinstance(queries, list):
                for q in queries:
                    new_pairs.append({
                        "query": q,
                        "pos": enhanced_content,
                        "chunk_id": chunk_id
                    })
                success = True
                processed_in_this_run += 1
                print(f"[{processed_in_this_run}/{LIMIT}] (Tài liệu: {doc_name}) Sinh thành công 3 câu hỏi.")
            else:
                attempts += 1
        except Exception as e:
            attempts += 1
            print(f"Lỗi ở chunk thứ {idx+1} (Thử lại {attempts}): {e}")
            time.sleep(5)

# Trộn dữ liệu cũ và mới để ghi file
all_pairs = existing_pairs + new_pairs
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(all_pairs, f, indent=2, ensure_ascii=False)

print(f"- Đã xử lý thêm: {processed_in_this_run} chunks mới trong lượt này.")
print(f"- Tổng số câu hỏi hiện có trong file '{OUTPUT_PATH}': {len(all_pairs)} câu hỏi.")

Đã nạp 33 API Key cho hệ thống.
Tìm thấy dữ liệu cũ: 1386 câu hỏi (đã xử lý 462 chunks).

Bắt đầu sinh câu hỏi...



[CẢNH BÁO] Key số 1 cạn kiệt Token (Lỗi 429). Chuyển sang dùng Key số 2...



[CẢNH BÁO] Key số 2 cạn kiệt Token (Lỗi 429). Chuyển sang dùng Key số 3...



[CẢNH BÁO] Key số 3 cạn kiệt Token (Lỗi 429). Chuyển sang dùng Key số 4...



[CẢNH BÁO] Key số 4 cạn kiệt Token (Lỗi 429). Chuyển sang dùng Key số 5...
[1/22] (Tài liệu: Unet++) Sinh thành công 3 câu hỏi.


...

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2539.93ms



[CẢNH BÁO] Key số 11 cạn kiệt Token (Lỗi 429). Chuyển sang dùng Key số 12...



[CẢNH BÁO] Key số 12 cạn kiệt Token (Lỗi 429). Chuyển sang dùng Key số 13...
[8/22] (Tài liệu: Unet) Sinh thành công 3 câu hỏi.
[9/22] (Tài liệu: Unet) Sinh thành công 3 câu hỏi.
[10/22] (Tài liệu: Unet) Sinh thành công 3 câu hỏi.
[11/22] (Tài liệu: Unet) Sinh thành công 3 câu hỏi.
[12/22] (Tài liệu: Unet) Sinh thành công 3 câu hỏi.



[CẢNH BÁO] Key số 13 cạn kiệt Token (Lỗi 429). Chuyển sang dùng Key số 14...
[13/22] (Tài liệu: Unet) Sinh thành công 3 câu hỏi.
[14/22] (Tài liệu: Unet) Sinh thành công 3 câu hỏi.
[15/22] (Tài liệu: Unet) Sinh thành công 3 câu hỏi.



[CẢNH BÁO] Key số 14 cạn kiệt Token (Lỗi 429). Chuyển sang dùng Key số 15...
[16/22] (Tài liệu: Unet) Sinh thành công 3 câu hỏi.
[17/22] (Tài liệu: Unet) Sinh thành công 3 câu hỏi.
- Đã xử lý thêm: 17 chunks mới trong lượt này.
- Tổng số câu hỏi hiện có trong file 'train_embedding_data.json': 1437 câu hỏi.


# Xem thử kết quả dữ liệu vừa sinh (Kiểm tra thủ công)

In [ ]:
import json

with open('train_embedding_data.json', 'r', encoding='utf-8') as f:
    preview_data = json.load(f)

print(f"Tổng số câu hỏi hiện tại: {len(preview_data)}")
print("\n--- Xem thử 5 câu hỏi đầu tiên ---")
for i, item in enumerate(preview_data[:5]):
    print(f"Câu hỏi {i+1}: {item['query']}")
    print(f"-> Chunk tương ứng (rút gọn): {item['pos'][:120]}...\n")

Tổng số câu hỏi hiện tại: 1437

--- Xem thử 5 câu hỏi đầu tiên ---
Câu hỏi 1: What is the purpose of the novel attention gate (AG) model in medical imaging as proposed in the Attention U-Net paper?
-> Chunk tương ứng (rút gọn): Paper: Att Unet Domain: CV Content: 8

2018

1

0

2

y a M 0 2 ] V C . s c [ 3 v 9 9 9 3 0 . 4 0 8 1

:

v

arXiv

i

X...

Câu hỏi 2: How does the Attention U-Net architecture improve prediction performance and computational efficiency when integrated into standard CNNs for multi-class image segmentation?
-> Chunk tương ứng (rút gọn): Paper: Att Unet Domain: CV Content: 8

2018

1

0

2

y a M 0 2 ] V C . s c [ 3 v 9 9 9 3 0 . 4 0 8 1

:

v

arXiv

i

X...

Câu hỏi 3: On what datasets was the Attention U-Net evaluated for medical image segmentation, and what were the findings regarding its performance with attention gates?
-> Chunk tương ứng (rút gọn): Paper: Att Unet Domain: CV Content: 8

2018

1

0

2

y a M 0 2 ] V C . s c [ 3 v 9 9 9 3 0 . 4 0 8 1

:

v



# Fine-tune mô hình Embedding

In [ ]:
import os
import json
import torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

# 1. Cấu hình phần cứng
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sử dụng thiết bị: {device}")

# 2. Đọc dữ liệu huấn luyện
train_data_path = 'train_embedding_data.json'
with open(train_data_path, 'r', encoding='utf-8') as f:
    train_pairs = json.load(f)

# 3. Định dạng dữ liệu
train_examples = []
for item in train_pairs:
    train_examples.append(InputExample(texts=[item['query'], item['pos']]))

# 4. DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

# 5. Khởi tạo mô hình cơ sở
model_name = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Đang nạp mô hình nền tảng: {model_name}...")
model = SentenceTransformer(model_name, device=device)

# 6. Loss function
train_loss = losses.MultipleNegativesRankingLoss(model=model)

# 7. Huấn luyện (Fine-tune)
num_epochs = 3
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1)

print(f"Bắt đầu huấn luyện...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    show_progress_bar=True
)

# 8. Lưu mô hình đã huấn luyện
output_model_path = 'finetuned-all-MiniLM-L6-v2'
model.save(output_model_path)
print(f"\nHoàn thành! Mô hình đã được lưu tại: {output_model_path}")

/tmp/ipykernel_778/2716953553.py:4: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


Sử dụng thiết bị: cuda
Đang nạp mô hình nền tảng: sentence-transformers/all-MiniLM-L6-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

Bắt đầu huấn luyện...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Hoàn thành! Mô hình đã được lưu tại: finetuned-all-MiniLM-L6-v2


# Nén và tải mô hình

In [ ]:
# Nén thư mục mô hình thành file zip
!zip -r finetuned_model.zip finetuned-all-MiniLM-L6-v2

# Tự động tải file zip
from google.colab import files
files.download('finetuned_model.zip')

  adding: finetuned-all-MiniLM-L6-v2/ (stored 0%)
  adding: finetuned-all-MiniLM-L6-v2/sentence_bert_config.json (deflated 43%)
  adding: finetuned-all-MiniLM-L6-v2/1_Pooling/ (stored 0%)
  adding: finetuned-all-MiniLM-L6-v2/1_Pooling/config.json (deflated 16%)
  adding: finetuned-all-MiniLM-L6-v2/2_Normalize/ (stored 0%)
  adding: finetuned-all-MiniLM-L6-v2/config.json (deflated 52%)
  adding: finetuned-all-MiniLM-L6-v2/modules.json (deflated 64%)
  adding: finetuned-all-MiniLM-L6-v2/model.safetensors (deflated 8%)
  adding: finetuned-all-MiniLM-L6-v2/config_sentence_transformers.json (deflated 40%)
  adding: finetuned-all-MiniLM-L6-v2/tokenizer.json (deflated 71%)
  adding: finetuned-all-MiniLM-L6-v2/README.md (deflated 63%)
  adding: finetuned-all-MiniLM-L6-v2/tokenizer_config.json (deflated 51%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>